In [ ]:
'Step 1: Imports'
from processing_layer.data_loader import load_data
from processing_layer.data_preprocessing import preprocess_pipeline
from demand_analysis_layer.demand_analysis import demand_analysis_pipeline
from optimization_layer.inventory_optimization import calculate_inventory_metrics

from query_layer.decision_queries import get_reorder_quantity
from query_layer.decision_queries import get_reorder_point
from query_layer.decision_queries import get_lead_time_demand

from query_layer.boolean_queries import get_stockout_risk
from query_layer.boolean_queries import get_overstock_risk

from query_layer.analytical_queries import get_safety_stock
from query_layer.analytical_queries import get_demand_trend

'Step 2: Load Data'
products, sales, purchases, inventory = load_data()

'Step 3: Preprocessing'
processed = preprocess_pipeline(sales, purchases)
daily_sales = processed["daily_sales"]
avg_demand = processed["avg_demand"]

'Step 4: Demand Analysis'
demand_results = demand_analysis_pipeline(daily_sales, avg_demand)
std_dev = demand_results["std_dev"]
final_demand = demand_results["final_demand"]
rolling_30 = demand_results["rolling_30"]

'Step 5: Optimization'
final_df = calculate_inventory_metrics(final_demand, std_dev, inventory)

'Step 6: (Important) Merge extra fields for queries'
final_df = final_df.merge(avg_demand, on="item_id")
final_df = final_df.merge(rolling_30, on="item_id")


In [7]:
'Step 7: Run Queries (Demo)'
item_id ='P001'

print("Reorder Quantity:", get_reorder_quantity(final_df, item_id))
print("Stockout Risk:", get_stockout_risk(final_df, item_id))
print("Safety Stock:", get_safety_stock(final_df, item_id))

Reorder Quantity: 21
Stockout Risk: YES
Safety Stock: 9.02


In [10]:
item_ids = ['P001', 'P005', 'P012']

for item in item_ids:
    print(f"\nItem {item}")
    
    print("Reorder Quantity:", get_reorder_quantity(final_df, item))
    print("Reorder Point:", get_reorder_point(final_df, item))
    print("Lead Time Demand:", get_lead_time_demand(final_df, item))
    
    print("Stockout Risk:", get_stockout_risk(final_df, item))
    print("Overstock Risk:", get_overstock_risk(final_df, item))
    
    print("Safety Stock:", get_safety_stock(final_df, item))
    print("Demand Trend:", get_demand_trend(final_df, item))


Item P001
Reorder Quantity: 21
Reorder Point: 39.44
Lead Time Demand: 30.41
Stockout Risk: YES
Overstock Risk: NO
Safety Stock: 9.02
Demand Trend: Increasing

Item P005
Reorder Quantity: 0
Reorder Point: 33.39
Lead Time Demand: 28.37
Stockout Risk: NO
Overstock Risk: YES
Safety Stock: 5.02
Demand Trend: Stable/Decreasing

Item P012
Reorder Quantity: 24
Reorder Point: 46.46
Lead Time Demand: 36.89
Stockout Risk: YES
Overstock Risk: NO
Safety Stock: 9.57
Demand Trend: Stable/Decreasing


In [14]:
print(final_df.isna().sum())

item_id                  0
final_demand             0
demand_std_dev           0
current_stock            0
lead_time_days           0
safety_stock             0
reorder_point            0
reorder_needed           0
reorder_quantity         0
avg_daily_demand         0
rolling_30_day_demand    0
dtype: int64
